# Graph RAG with Neo4j — vector search **+** knowledge graph

This notebook builds a **Graph RAG** pipeline over the threat-modeling reference
books using **Neo4j** as both the **vector database** (vector index over chunk
embeddings) *and* the **knowledge graph** (typed nodes + relationships).

Plain vector RAG retrieves the *k* chunks closest to the query and stops. **Graph
RAG** uses those vector hits as **entry points**, then **traverses the graph** to
pull in connected context the embedding alone would miss — the surrounding
passage, and other chunks about the same entities.

```
                    ┌──────────────── Neo4j ────────────────┐
 query ──encode──▶  │  VECTOR INDEX           GRAPH          │
                    │  (:Chunk {embedding})   (:Book)-[:HAS_CHUNK]->(:Chunk)
                    │      │ top-k seeds       (:Chunk)-[:NEXT]->(:Chunk)
                    │      ▼                   (:Chunk)-[:MENTIONS]->(:Entity)
                    │   seed chunks ──expand──▶ neighbors + same-entity chunks
                    └───────────────────────────────│──────┘
                                                     ▼
                                       grounded context ──▶ (optional) LLM answer
```

**Graph schema**

| Node | Key props | Relationships |
|------|-----------|----------------|
| `:Book` | `title` | `-[:HAS_CHUNK]->(:Chunk)` |
| `:Chunk` | `id, text, page, seq, embedding(384)` | `-[:NEXT]->(:Chunk)`, `-[:MENTIONS]->(:Entity)` |
| `:Entity` | `name, type` | `<-[:MENTIONS]-(:Chunk)` |

## Prerequisites

**1. A running Neo4j** (5.15+, for the GA vector index). Local Docker:
```bash
docker run -d --name neo4j-graphrag -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/graphrag-demo-pw neo4j:5.26
```
Browser UI: http://localhost:7474 · Bolt: `bolt://localhost:7687`. Override the
connection with `NEO4J_URI` / `NEO4J_USER` / `NEO4J_PASSWORD` env vars (or a
`.env`) — e.g. to point at a Neo4j Aura instance instead.

**2. Python deps:** `neo4j`, `sentence-transformers`, `torch`, `pypdf`,
`langchain-text-splitters` (GPU auto-detected for embeddings). `openai` is
optional (final answer step).

**3. The books** in a `Reference/Books` folder (auto-discovered upward), the same
ones `ingest_references.py` uses.

In [12]:
import os, time, logging
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()
logging.basicConfig(level="INFO", format="%(levelname)-7s | %(message)s")
for noisy in ("httpx", "httpcore", "urllib3", "huggingface_hub", "filelock",
              "sentence_transformers", "neo4j", "neo4j.pool", "neo4j.io"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "graphrag-demo-pw")

from neo4j import GraphDatabase
# notifications_min_severity='OFF' silences server-side "type/property may not exist"
# notices (harmless, but noisy on a graph we're still building).
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD),
                              notifications_min_severity="OFF")
driver.verify_connectivity()

def run(cypher, **params):
    """Run a Cypher statement and return rows as list[dict]."""
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, **params)]

ver = run("CALL dbms.components() YIELD name, versions, edition "
          "RETURN name, versions[0] AS version, edition")[0]
print(f"Connected to {NEO4J_URI} — {ver['name']} {ver['version']} ({ver['edition']})")

Connected to bolt://localhost:7687 — Neo4j Kernel 5.26.26 (community)


## 1. Load & chunk the source books
Read each PDF page → text → overlapping chunks (same recipe as the app's ingest).
Scanned books need a text layer first (the app OCRs them on ingest); here we just
read whatever text is present.

In [13]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def resolve_books_dir():
    env = os.getenv("REFERENCE_BOOKS_DIR")
    if env and Path(env).is_dir():
        return Path(env)
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "Reference" / "Books"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("Reference/Books not found; set REFERENCE_BOOKS_DIR.")

BOOKS_DIR = resolve_books_dir()
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)

def load_chunks(pdf_path):
    # Tolerate the saved-multipart-upload wrapper some PDFs have.
    raw = pdf_path.read_bytes()
    if raw[:5] != b"%PDF-":
        start = raw.find(b"%PDF-"); end = raw.rfind(b"%%EOF")
        import io
        stream = io.BytesIO(raw[start: end + 5] if end != -1 else raw[start:])
    else:
        import io; stream = io.BytesIO(raw)
    reader = PdfReader(stream)
    title = pdf_path.stem
    out, seq = [], 0
    for page_no, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        for j, piece in enumerate(splitter.split_text(text)):
            out.append({"id": f"{title}-p{page_no:04d}-c{j:02d}", "book": title,
                        "page": page_no, "seq": seq, "text": piece})
            seq += 1
    return out

chunks = []
for pdf in sorted(BOOKS_DIR.glob("*.pdf")):
    c = load_chunks(pdf)
    print(f"{pdf.name:45} -> {len(c):5d} chunks")
    chunks.extend(c)
print(f"\nTotal: {len(chunks)} chunks from {len(set(c['book'] for c in chunks))} books")

AutomotiveThreatModeling.pdf                  ->   869 chunks
threat_modeling_designing_for_security.pdf    ->  1932 chunks

Total: 2801 chunks from 2 books


## 2. Encode the chunks (local GPU embeddings)
Embed every chunk with `bge-small-en-v1.5` — 384-dim, L2-normalized so cosine
similarity is a dot product. These vectors are stored **on the `:Chunk` nodes**
and indexed by Neo4j.

In [14]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
EMB_MODEL = "BAAI/bge-small-en-v1.5"
print("device:", DEVICE, "| model:", EMB_MODEL)

embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)

t0 = time.time()
vectors = embedder.encode([c["text"] for c in chunks], batch_size=256,
                          normalize_embeddings=True, convert_to_numpy=True,
                          show_progress_bar=True)
for c, v in zip(chunks, vectors):
    c["embedding"] = v.tolist()
EMB_DIM = int(vectors.shape[1])
print(f"Embedded {len(chunks)} chunks (dim={EMB_DIM}) in {time.time()-t0:.1f}s on {DEVICE}")

def embed_query(text):
    q = "Represent this sentence for searching relevant passages: " + text
    return embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()

device: cuda | model: BAAI/bge-small-en-v1.5


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:01<00:00,  5.98it/s]

Embedded 2801 chunks (dim=384) in 1.9s on cuda


## 3. Build the knowledge graph
Wipe any prior demo graph, set uniqueness constraints, then bulk-load `:Book` and
`:Chunk` nodes (with embeddings) via batched `UNWIND`.

In [15]:
# Fresh start for this demo (only touches our labels).
run("MATCH (n) WHERE n:Book OR n:Chunk OR n:Entity DETACH DELETE n")
run("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE")
run("CREATE CONSTRAINT book_title IF NOT EXISTS FOR (b:Book) REQUIRE b.title IS UNIQUE")
run("CREATE CONSTRAINT entity_name IF NOT EXISTS FOR (e:Entity) REQUIRE e.name IS UNIQUE")

LOAD = """
UNWIND $rows AS row
MERGE (b:Book {title: row.book})
MERGE (c:Chunk {id: row.id})
  SET c.book = row.book, c.text = row.text, c.page = row.page,
      c.seq = row.seq, c.embedding = row.embedding
MERGE (b)-[:HAS_CHUNK]->(c)
"""
BATCH = 500
for i in range(0, len(chunks), BATCH):
    run(LOAD, rows=chunks[i:i+BATCH])
print("Loaded nodes:", run("MATCH (c:Chunk) RETURN count(c) AS n")[0]["n"], "chunks,",
      run("MATCH (b:Book) RETURN count(b) AS n")[0]["n"], "books")

Loaded nodes: 2801 chunks, 2 books


### 3a. `NEXT` relationships — reading order
Link consecutive chunks within each book so a vector hit can pull its **immediate
neighbours** (the passage it was cut from).

In [16]:
run("""
MATCH (b:Book)-[:HAS_CHUNK]->(c:Chunk)
WITH c ORDER BY c.book, c.seq
WITH collect(c) AS cs
UNWIND range(0, size(cs)-2) AS i
WITH cs[i] AS a, cs[i+1] AS b
WHERE a.book = b.book
MERGE (a)-[:NEXT]->(b)
""")
print("NEXT edges:", run("MATCH ()-[r:NEXT]->() RETURN count(r) AS n")[0]["n"])

NEXT edges: 2799


### 3b. `MENTIONS` relationships — the cross-links that make it a *graph*
A lightweight, deterministic entity pass: for a curated threat-modeling vocabulary
(STRIDE categories, standards, core concepts), link any chunk that mentions a term
to an `:Entity` node. These edges connect chunks **across books and pages** that
share a topic — the paths Graph RAG traverses.

In [17]:
VOCAB = {
    # STRIDE
    "Spoofing": "stride", "Tampering": "stride", "Repudiation": "stride",
    "Information Disclosure": "stride", "Denial of Service": "stride",
    "Elevation of Privilege": "stride",
    # standards / frameworks
    "ISO/SAE 21434": "standard", "ISO 21434": "standard", "ISO 27001": "standard",
    "ISO 31000": "standard", "STRIDE": "method", "DREAD": "method",
    "attack tree": "concept", "data flow diagram": "concept",
    "trust boundary": "concept", "threat model": "concept",
    "mitigation": "concept", "asset": "concept", "vulnerability": "concept",
}

mentions = []  # (chunk_id, entity_name, entity_type)
for c in chunks:
    low = c["text"].lower()
    seen = set()
    for term, etype in VOCAB.items():
        name = term.lower()
        if name in low and name not in seen:
            seen.add(name)
            mentions.append({"cid": c["id"], "name": term, "type": etype})

run("""
UNWIND $rows AS row
MATCH (c:Chunk {id: row.cid})
MERGE (e:Entity {name: row.name}) SET e.type = row.type
MERGE (c)-[:MENTIONS]->(e)
""", rows=mentions)

print("MENTIONS edges:", len(mentions))
print("\nTop entities by chunk coverage:")
for r in run("""MATCH (e:Entity)<-[:MENTIONS]-(c:Chunk)
               RETURN e.name AS entity, e.type AS type, count(c) AS chunks
               ORDER BY chunks DESC LIMIT 10"""):
    print(f"  {r['entity']:24} {r['type']:9} {r['chunks']:5d} chunks")

MENTIONS edges: 2457

Top entities by chunk coverage:
  threat model             concept     668 chunks
  asset                    concept     307 chunks
  mitigation               concept     209 chunks
  STRIDE                   method      201 chunks
  Tampering                stride      172 chunks
  ISO/SAE 21434            standard    159 chunks
  vulnerability            concept     157 chunks
  Repudiation              stride      116 chunks
  Information Disclosure   stride      112 chunks
  Elevation of Privilege   stride      106 chunks


## 4. Create the vector index
Neo4j's native vector index over `:Chunk.embedding` (cosine, 384-dim) powers
approximate-nearest-neighbour search with `db.index.vector.queryNodes`.

In [18]:
run(f"""
CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS
FOR (c:Chunk) ON c.embedding
OPTIONS {{ indexConfig: {{
  `vector.dimensions`: {EMB_DIM},
  `vector.similarity_function`: 'cosine'
}} }}
""")
run("CALL db.awaitIndex('chunk_embeddings', 300)")   # block until ONLINE
idx = run("SHOW INDEXES YIELD name, type, state WHERE name='chunk_embeddings' "
          "RETURN name, type, state")[0]
print("Vector index:", idx)

Vector index: {'name': 'chunk_embeddings', 'type': 'VECTOR', 'state': 'ONLINE'}


## 5. Graph RAG retrieval — vector seed → graph expansion
First the **plain vector** baseline (entry points), then **expand** through the
graph: each seed contributes its `NEXT`/previous neighbours (local context) and
other chunks that `MENTIONS` the same entities (topically related, corpus-wide).

In [19]:
QUERY = "How do I mitigate spoofing threats when designing authentication?"
SEED_K = 5

qvec = embed_query(QUERY)
seeds = run("""
CALL db.index.vector.queryNodes('chunk_embeddings', $k, $qvec)
YIELD node, score
RETURN node.id AS id, node.book AS book, node.page AS page, score, node.text AS text
ORDER BY score DESC
""", k=SEED_K, qvec=qvec)

print(f"QUERY: {QUERY}\n\n--- Vector seeds (plain RAG) ---")
for i, s in enumerate(seeds, 1):
    print(f"[{i}] {s['book']} p.{s['page']}  score={s['score']:.4f}  "
          f"{' '.join(s['text'].split())[:90]}…")

QUERY: How do I mitigate spoofing threats when designing authentication?

--- Vector seeds (plain RAG) ---
[1] threat_modeling_designing_for_security p.48  score=0.9331  to address these threats, see Chapters 8 and 9, “Defensive Building Blocks” and “Tradeoffs…
[2] threat_modeling_designing_for_security p.201  score=0.9075  Chapter 8 ■ Defensive Tactics and Technologies 165 c08.indd 12:26:46:PM 01/13/2014 Page 16…
[3] threat_modeling_designing_for_security p.100  score=0.9004  64 Part II ■ Finding Threats c03.indd 07:51:54:AM 01/15/2014 Page 64 by putting a program …
[4] threat_modeling_designing_for_security p.382  score=0.9002  for a MITM or other attacker to wedge themselves in. Therefore, make a deci- sion about wh…
[5] threat_modeling_designing_for_security p.14  score=0.8993  xii Contents ftoc.indd 11:56:7:AM 01/17 /2014 Page xii Chapter 8 Defensive Tactics and Tec…


In [20]:
# Graph expansion: from the seed chunks, gather neighbours + same-entity chunks.
seed_ids = [s["id"] for s in seeds]
expanded = run("""
MATCH (seed:Chunk) WHERE seed.id IN $ids
// (a) adjacent passages in reading order (either direction)
OPTIONAL MATCH (seed)-[:NEXT]-(nbr:Chunk)
// (b) other chunks mentioning the same entities
OPTIONAL MATCH (seed)-[:MENTIONS]->(e:Entity)<-[:MENTIONS]-(rel:Chunk)
WITH seed,
     collect(DISTINCT nbr) AS nbrs,
     collect(DISTINCT {chunk: rel, entity: e.name}) AS rels
UNWIND ([{chunk: seed, via: 'seed', entity: null}]
      + [n IN nbrs | {chunk: n, via: 'NEXT (neighbour)', entity: null}]
      + [r IN rels WHERE r.chunk IS NOT NULL |
            {chunk: r.chunk, via: 'MENTIONS', entity: r.entity}]) AS item
WITH item.chunk AS c, item.via AS via, item.entity AS entity
WHERE c IS NOT NULL
RETURN DISTINCT c.id AS id, c.book AS book, c.page AS page,
       head(collect(via)) AS via, head(collect(entity)) AS via_entity,
       c.text AS text
LIMIT 25
""", ids=seed_ids)

import pandas as pd
pd.set_option("display.max_colwidth", 80)
exp_df = pd.DataFrame([{
    "book": e["book"], "page": e["page"], "reached_via": e["via"],
    "entity": e["via_entity"] or "", "snippet": " ".join(e["text"].split())[:80],
} for e in expanded])
print(f"Vector seeds: {len(seeds)}  →  graph-expanded context: {len(expanded)} chunks")
exp_df

Vector seeds: 5  →  graph-expanded context: 25 chunks


,book,page,reached_via,entity,snippet
0,threat_modeling_designing_for_security,14,seed,Tampering,xii Contents ftoc.indd 11:56:7:AM 01/17 /2014 Page xii Chapter 8 Defensive T...
1,threat_modeling_designing_for_security,13,NEXT (neighbour),,Starting the Threat Modeling Project 126 When to Threat Model 126 What to St...
2,threat_modeling_designing_for_security,14,NEXT (neighbour),,Classic Strategies for Risk Management 168 Avoiding Risks 168 Addressing Ris...
3,AutomotiveThreatModeling,185,MENTIONS,Tampering,(no cryptographic signature verification) w Compromised cybersecurity proper...
4,threat_modeling_designing_for_security,478,MENTIONS,Tampering,■ Local admin ■ Call chain ■ Caller ■ Callee ■ Spoof an external entity ■ Su...
5,threat_modeling_designing_for_security,554,MENTIONS,Tampering,"popular account management tools, each of which presents a tampering threat...."
6,threat_modeling_designing_for_security,182,MENTIONS,Tampering,"dencies are covered under tampering); but in general, only programs running ..."
7,threat_modeling_designing_for_security,498,MENTIONS,Tampering,"memory When you release storage, does the OS clean it? As above None Backup ..."
8,threat_modeling_designing_for_security,46,MENTIONS,Tampering,10 Part I 0 ■ Getting Starte d ■ Tampering is modifying something you’re not...
9,threat_modeling_designing_for_security,554,MENTIONS,Tampering,"ous requests. If logs are stored on the same system as the main database, th..."


## 6. (Optional) Answer with the graph-grounded context
If `OPENAI_API_KEY` is set, hand the expanded context to an LLM with a
cite-your-sources instruction. Otherwise this step is skipped.

In [21]:
USE_LLM = bool(os.getenv("OPENAI_API_KEY"))

def build_context(rows, max_chars=4000):
    parts, total = [], 0
    for r in rows:
        block = f"[{r['book']} p.{r['page']}] " + " ".join(r['text'].split())
        if total + len(block) > max_chars:
            break
        parts.append(block); total += len(block)
    return "\n\n".join(parts)

if USE_LLM:
    from openai import OpenAI
    client = OpenAI()
    context = build_context(expanded)
    msg = [
        {"role": "system", "content": "You answer using ONLY the provided context. "
         "Cite sources as [book p.N]. If the context is insufficient, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {QUERY}"},
    ]
    resp = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), messages=msg,
        temperature=0.2, max_tokens=400)
    print(resp.choices[0].message.content)
else:
    print("OPENAI_API_KEY not set — skipping LLM answer. The graph-expanded context "
          "above is what would be sent as grounding.")

To mitigate spoofing threats when designing authentication, you can implement the following tactics:

1. **Strong Authentication Mechanisms**: Use multi-factor authentication (MFA) to ensure that users must provide multiple forms of verification before gaining access.

2. **Secure Password Policies**: Enforce strong password requirements and encourage regular password changes to reduce the risk of unauthorized access.

3. **Session Management**: Implement secure session management practices, such as using secure cookies and session timeouts, to prevent session hijacking.

4. **User Education**: Educate users about phishing attacks and the importance of verifying the authenticity of communications before providing sensitive information.

5. **Monitoring and Logging**: Continuously monitor authentication attempts and log access to detect and respond to suspicious activities promptly.

These strategies help to enhance the security of the authentication process and reduce the risk of spoof

## 7. Reusable `graph_rag()` helper

In [22]:
def graph_rag(query, seed_k=5, limit=25):
    qv = embed_query(query)
    seeds = run("""CALL db.index.vector.queryNodes('chunk_embeddings', $k, $qvec)
                   YIELD node, score RETURN node.id AS id, score""", k=seed_k, qvec=qv)
    ids = [s["id"] for s in seeds]
    ctx = run("""
        MATCH (seed:Chunk) WHERE seed.id IN $ids
        OPTIONAL MATCH (seed)-[:NEXT]-(nbr:Chunk)
        OPTIONAL MATCH (seed)-[:MENTIONS]->(:Entity)<-[:MENTIONS]-(rel:Chunk)
        WITH collect(DISTINCT seed) + collect(DISTINCT nbr) + collect(DISTINCT rel) AS cs
        UNWIND cs AS c WITH c WHERE c IS NOT NULL
        RETURN DISTINCT c.book AS book, c.page AS page, c.text AS text LIMIT $limit
    """, ids=ids, limit=limit)
    return {"seeds": seeds, "context": ctx}

demo = graph_rag("What is an attack tree and how is it used?")
print("seeds:", len(demo["seeds"]), "| expanded context chunks:", len(demo["context"]))
for r in demo["context"][:4]:
    print(f"  [{r['book']} p.{r['page']}] {' '.join(r['text'].split())[:90]}…")

seeds: 5 | expanded context chunks: 25
  [threat_modeling_designing_for_security p.123] 87 c04.indd 11:38:53:AM 01/17 /2014 Page 87 As Bruce Schneier wrote in his introduction to…
  [threat_modeling_designing_for_security p.136] which means that understanding a new tree takes longer. Summary Attack trees fi t well int…
  [threat_modeling_designing_for_security p.136] Attack trees can be represented as graphical trees, as outlines, or in software. You saw a…
  [threat_modeling_designing_for_security p.124] 88 Part II ■ Finding Threats c04.indd 11:38:53:AM 01/17 /2014 Page 88 a tree to help you t…


## Notes & cleanup

- **Why this beats plain vector RAG:** the vector index finds *semantically*
  similar chunks; the graph adds chunks that are *structurally* related — the
  surrounding passage (`NEXT`) and corpus-wide chunks on the same entity
  (`MENTIONS`). That extra context is what an embedding-only search drops.
- **Schema is the lever.** Here entities come from a keyword vocabulary; in a real
  system you'd extract them with an LLM (or `neo4j-graphrag` / LangChain's
  `LLMGraphTransformer`) and add richer relationship types to traverse.
- **Neo4j as vector DB:** `db.index.vector.queryNodes` does the ANN search; the
  same database then answers graph queries — no separate vector store to sync.
- **Tuning:** raise `SEED_K` for more entry points; cap expansion with `LIMIT`;
  rerank the expanded set (see `reference_query_rerank.ipynb`) before the LLM.

**Stop / remove the demo database when done:**
```bash
docker stop neo4j-graphrag      # keep data
docker rm -f neo4j-graphrag     # delete container
```

In [23]:
# Sanity check + tidy up the driver
stats = {
    "chunks":  run("MATCH (c:Chunk) RETURN count(c) AS n")[0]["n"],
    "entities": run("MATCH (e:Entity) RETURN count(e) AS n")[0]["n"],
    "NEXT":     run("MATCH ()-[r:NEXT]->() RETURN count(r) AS n")[0]["n"],
    "MENTIONS": run("MATCH ()-[r:MENTIONS]->() RETURN count(r) AS n")[0]["n"],
}
print("graph:", stats)
print("OK — encode → vector seed → graph expansion → (optional) answer ran end-to-end.")
driver.close()

graph: {'chunks': 2801, 'entities': 18, 'NEXT': 2799, 'MENTIONS': 2457}
OK — encode → vector seed → graph expansion → (optional) answer ran end-to-end.
